# 1 执行顺序总结（从上到下）：
### FROM
### JOIN
### WHERE
### GROUP BY
### HAVING
### SELECT
### DISTINCT
### "Window Function"
### ORDER BY
### LIMIT / OFFSET

# 2 SQL 查询执行顺序示例分析

我们通过一个具体的例子来说明 SQL 查询的执行顺序。假设我们有一个关于订单的表 `orders`，包含以下字段：

- `order_id`：订单ID
- `customer_id`：客户ID
- `order_date`：订单日期
- `order_amount`：订单金额
- `product_id`：产品ID

### 原始 `orders` 表（数据示例）：

| order_id | customer_id | order_date  | order_amount | product_id |
|----------|-------------|-------------|--------------|------------|
| 1        | 1           | 2024-01-01  | 100          | A          |
| 2        | 1           | 2024-01-01  | 150          | B          |
| 3        | 2           | 2024-01-01  | 200          | A          |
| 4        | 1           | 2024-01-02  | 250          | C          |
| 5        | 2           | 2024-01-02  | 300          | B          |
| 6        | 3           | 2024-01-02  | 350          | A          |
| 7        | 2           | 2024-01-03  | 100          | C          |
| 8        | 3           | 2024-01-03  | 200          | A          |

### 原始 `products` 表（数据示例）：

| product\_id | product\_name |
| ----------- | ------------- |
| A           | Product A     |
| B           | Product B     |
| C           | Product C     |

* `product_id`：产品ID
* `product_name`：产品名称

我们将通过 `JOIN` 将 `orders` 表和 `products` 表连接起来，查询每个客户在 2024 年 1 月 1 日之后的订单金额总和，并计算每个客户的累积订单金额，按日期排序，去重，最终结果限制为 5 条记录。

### 更新的查询：

```sql
SELECT 
    o.customer_id,
    o.order_date,
    SUM(o.order_amount) AS total_order_amount,
    SUM(o.order_amount) OVER (PARTITION BY o.customer_id ORDER BY o.order_date) AS running_total,
    p.product_name
FROM 
    orders o
JOIN 
    products p ON o.product_id = p.product_id
WHERE 
    o.order_date >= '2024-01-01'
GROUP BY 
    o.customer_id, o.order_date, p.product_name
HAVING 
    SUM(o.order_amount) > 150
ORDER BY 
    o.customer_id, o.order_date
LIMIT 5;
```

### 执行过程分析：

#### 1. **FROM**

数据源是 `orders` 表（`o` 的别名）。
初始数据集为 `orders` 表中的所有记录：

| order\_id | customer\_id | order\_date | order\_amount | product\_id |
| --------- | ------------ | ----------- | ------------- | ----------- |
| 1         | 1            | 2024-01-01  | 100           | A           |
| 2         | 1            | 2024-01-01  | 150           | B           |
| 3         | 2            | 2024-01-01  | 200           | A           |
| 4         | 1            | 2024-01-02  | 250           | C           |
| 5         | 2            | 2024-01-02  | 300           | B           |
| 6         | 3            | 2024-01-02  | 350           | A           |
| 7         | 2            | 2024-01-03  | 100           | C           |
| 8         | 3            | 2024-01-03  | 200           | A           |

#### 2. **ON**

`JOIN` 条件在这里是 `o.product_id = p.product_id`，连接 `orders` 表和 `products` 表。`JOIN` 会根据 `product_id` 字段把 `orders` 表和 `products` 表中的数据匹配在一起。

连接后的数据集（展示部分）：

| order\_id | customer\_id | order\_date | order\_amount | product\_id | product\_name |
| --------- | ------------ | ----------- | ------------- | ----------- | ------------- |
| 1         | 1            | 2024-01-01  | 100           | A           | Product A     |
| 2         | 1            | 2024-01-01  | 150           | B           | Product B     |
| 3         | 2            | 2024-01-01  | 200           | A           | Product A     |
| 4         | 1            | 2024-01-02  | 250           | C           | Product C     |
| 5         | 2            | 2024-01-02  | 300           | B           | Product B     |
| 6         | 3            | 2024-01-02  | 350           | A           | Product A     |
| 7         | 2            | 2024-01-03  | 100           | C           | Product C     |
| 8         | 3            | 2024-01-03  | 200           | A           | Product A     |

#### 3. **WHERE**

对数据进行过滤，只保留订单日期大于等于 2024 年 1 月 1 日的记录。

过滤后的数据：

| order\_id | customer\_id | order\_date | order\_amount | product\_id | product\_name |
| --------- | ------------ | ----------- | ------------- | ----------- | ------------- |
| 1         | 1            | 2024-01-01  | 100           | A           | Product A     |
| 2         | 1            | 2024-01-01  | 150           | B           | Product B     |
| 3         | 2            | 2024-01-01  | 200           | A           | Product A     |
| 4         | 1            | 2024-01-02  | 250           | C           | Product C     |
| 5         | 2            | 2024-01-02  | 300           | B           | Product B     |
| 6         | 3            | 2024-01-02  | 350           | A           | Product A     |
| 7         | 2            | 2024-01-03  | 100           | C           | Product C     |
| 8         | 3            | 2024-01-03  | 200           | A           | Product A     |

#### 4. **GROUP BY**

数据按 `customer_id`、`order_date` 和 `product_name` 进行分组，计算每个客户每个日期每个产品的订单总金额。

分组后结果：

| customer\_id | order\_date | product\_name | total\_order\_amount |
| ------------ | ----------- | ------------- | -------------------- |
| 1            | 2024-01-01  | Product A     | 100                  |
| 1            | 2024-01-01  | Product B     | 150                  |
| 2            | 2024-01-01  | Product A     | 200                  |
| 1            | 2024-01-02  | Product C     | 250                  |
| 2            | 2024-01-02  | Product B     | 300                  |
| 3            | 2024-01-02  | Product A     | 350                  |
| 2            | 2024-01-03  | Product C     | 100                  |
| 3            | 2024-01-03  | Product A     | 200                  |

#### 5. **HAVING**

过滤掉 `total_order_amount` 小于或等于 150 的分组。

过滤后的数据：

| customer\_id | order\_date | product\_name | total\_order\_amount |
| ------------ | ----------- | ------------- | -------------------- |
| 1            | 2024-01-01  | Product A     | 100                  |
| 1            | 2024-01-01  | Product B     | 150                  |
| 2            | 2024-01-01  | Product A     | 200                  |
| 1            | 2024-01-02  | Product C     | 250                  |
| 2            | 2024-01-02  | Product B     | 300                  |
| 3            | 2024-01-02  | Product A     | 350                  |
| 3            | 2024-01-03  | Product A     | 200                  |

#### 6. **SELECT**

提取 `customer_id`、`order_date`、`total_order_amount`，并计算窗口函数 `running_total`，即每个客户按日期排序的累积订单金额。

执行结果：

| customer\_id | order\_date | product\_name | total\_order\_amount | running\_total |
| ------------ | ----------- | ------------- | -------------------- | -------------- |
| 1            | 2024-01-01  | Product A     | 100                  | 100            |
| 1            | 2024-01-01  | Product B     | 150                  | 250            |
| 2            | 2024-01-01  | Product A     | 200                  | 200            |
| 1            | 2024-01-02  | Product C     | 250                  | 500            |
| 2            | 2024-01-02  | Product B     | 300                  | 500            |
| 3            | 2024-01-02  | Product A     | 350                  | 350            |
| 3            | 2024-01-03  | Product A     | 200                  | 550            |

#### 7. **DISTINCT**

没有 `DISTINCT` 操作影响，这一步直接跳过。

#### 8. **ORDER BY**

按 `customer_id` 和 `order_date` 排序。

排序后的结果：

| customer\_id | order\_date | product\_name | total\_order\_amount | running\_total |
| ------------ | ----------- | ------------- | -------------------- | -------------- |
| 1            | 2024-01-01  | Product A     | 100                  | 100            |
| 1            | 2024-01-01  | Product B     | 150                  | 250            |
| 2            | 2024-01-01  | Product A     | 200                  | 200            |
| 1            | 2024-01-02  | Product C     | 250                  | 500            |
| 2            | 2024-01-02  | Product B     | 300                  | 500            |

#### 9. **LIMIT / OFFSET**

限制返回 5 条记录。

最终结果：

| customer\_id | order\_date | product\_name | total\_order\_amount | running\_total |
| ------------ | ----------- | ------------- | -------------------- | -------------- |
| 1            | 2024-01-01  | Product A     | 100                  | 100            |
| 1            | 2024-01-01  | Product B     | 150                  | 250            |
| 2            | 2024-01-01  | Product A     | 200                  | 200            |
| 1            | 2024-01-02  | Product C     | 250                  | 500            |
| 2            | 2024-01-02  | Product B     | 300                  | 500            |

### 总结

### 通过加入 `JOIN` 操作，我们把 `orders` 表和 `products` 表连接起来，查询了每个客户每个日期和每个产品的订单总金额，并计算了每个客户的累积订单金额。


# 3 主键 外键 索引

3.1 主键（Primary Key）
* 定义：主键是用于唯一标识表中每一行数据的字段（或字段组合）。一个表只能有一个主键。

3.2 外键（Foreign Key）
* 定义：外键是表与表之间的联系。它是在一个表中引用另一个表的主键字段，用来建立两个表之间的关系。

3.3 索引（Index）
* 定义：索引是数据库中的一种数据结构，它提高了查询的效率，类似于书籍的目录

```sql

CREATE TABLE customers (
    customer_id INT PRIMARY KEY,  -- 主键
    customer_name VARCHAR(100)
);

CREATE TABLE orders (
    order_id INT PRIMARY KEY,  -- 主键
    order_date DATE,
    customer_id INT,  -- 外键
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)  -- 外键约束
);
```

在这个例子中：

* customers 表中的 customer_id 是主键，它唯一标识每一个客户。
* orders 表中的 customer_id 是外键，它引用 customers 表中的 customer_id，确保每个订单都与一个存在的客户关联。

# 4 数据表连接方式

## 4.1  UNION 和 UNION ALL

UNION
* 会自动去重，只保留唯一的记录。
* 可能导致性能下降（因为要去重排序）。

UNION ALL
* 不去重，保留所有记录，包括重复项。
* 执行速度比 UNION 快，因为不需要做去重处理。


### 📋 表数据

#### 表 A：

| name  |
| ----- |
| Alice |
| Bob   |

#### 表 B：

| name  |
| ----- |
| Bob   |
| Carol |


### 🔀 SQL 语句对比

#### `UNION`：自动去重

```sql
SELECT name FROM A
UNION
SELECT name FROM B;
```

🔽 **结果：**

| name  |
| ----- |
| Alice |
| Bob   |
| Carol |


#### `UNION ALL`：保留重复

```sql
SELECT name FROM A
UNION ALL
SELECT name FROM B;
```

🔽 **结果：**

| name  |
| ----- |
| Alice |
| Bob   |
| Bob   |
| Carol |



# 4.2 内连接 左连接 右连接 全连接

### 假设我们有以下两张表：

#### 表 A（部门）：

| dept\_id | dept\_name |
| -------- | ---------- |
| 1        | HR         |
| 2        | IT         |
| 3        | Sales      |

#### 表 B（员工）：

| emp\_id | emp\_name | dept\_id |
| ------- | --------- | -------- |
| 101     | Alice     | 1        |
| 102     | Bob       | 2        |
| 103     | Carol     | 4        |


## 1️⃣ `INNER JOIN`（内连接）

只返回 **两表中匹配的记录**（即 dept\_id 相等的记录）。

```sql
SELECT A.dept_name, B.emp_name
FROM A
INNER JOIN B ON A.dept_id = B.dept_id;
```

📌 **结果：**

| dept\_name | emp\_name |
| ---------- | --------- |
| HR         | Alice     |
| IT         | Bob       |


## 2️⃣ `LEFT JOIN`（左连接）

返回左表（A）**所有记录**，右表（B）中没有匹配的，用 `NULL` 填充。

```sql
SELECT A.dept_name, B.emp_name
FROM A
LEFT JOIN B ON A.dept_id = B.dept_id;
```

📌 **结果：**

| dept\_name | emp\_name |
| ---------- | --------- |
| HR         | Alice     |
| IT         | Bob       |
| Sales      | NULL      |


## 3️⃣ `RIGHT JOIN`（右连接）

返回右表（B）**所有记录**，左表（A）中没有匹配的，用 `NULL` 填充。

```sql
SELECT A.dept_name, B.emp_name
FROM A
RIGHT JOIN B ON A.dept_id = B.dept_id;
```

📌 **结果：**

| dept\_name | emp\_name |
| ---------- | --------- |
| HR         | Alice     |
| IT         | Bob       |
| NULL       | Carol     |


## 4️⃣ `OUTER JOIN`（全连接）

返回左表和右表的**所有记录**，不匹配的用 `NULL` 填充。

```sql
SELECT A.dept_name, B.emp_name
FROM A
OUTER JOIN B ON A.dept_id = B.dept_id;
```

📌 **结果：**

| dept\_name | emp\_name |
| ---------- | --------- |
| HR         | Alice     |
| IT         | Bob       |
| Sales      | NULL      |
| NULL       | Carol     |


### ✅ 总结对比表

| 类型              | 保留左表   | 保留右表   | 匹配条件 | 不匹配用 NULL |
| --------------- | ------ | ------ | ---- | --------- |
| INNER JOIN      | ✅      | ✅      | ✅    | ❌         |
| LEFT JOIN       | ✅      | ✅（仅匹配） | ✅    | ✅（右表）     |
| RIGHT JOIN      | ✅（仅匹配） | ✅      | ✅    | ✅（左表）     |
| FULL OUTER JOIN | ✅      | ✅      | ✅    | ✅（左右都填）   |




# 5 SQL 常见函数


## 5.1 聚合函数（Aggregate Functions）

| 函数        | 作用     | 示例                                |
| --------- | ------ | --------------------------------- |
| `COUNT()` | 统计记录数量 | `COUNT(*)` / `COUNT(DISTINCT id)` |
| `SUM()`   | 求和     | `SUM(salary)`                     |
| `AVG()`   | 平均值    | `AVG(score)`                      |
| `MIN()`   | 最小值    | `MIN(age)`                        |
| `MAX()`   | 最大值    | `MAX(order_date)`                 |


## 5.2 排序/窗口函数（Ranking & Window Functions）


## 示例数据表：sales（销售记录）

| id | employee | department | sales | sale\_date |
| -- | -------- | ---------- | ----- | ---------- |
| 1  | Alice    | East       | 100   | 2024-01-01 |
| 2  | Bob      | East       | 200   | 2024-01-02 |
| 3  | Carol    | West       | 300   | 2024-01-01 |
| 4  | David    | East       | 150   | 2024-01-03 |
| 5  | Eva      | West       | 250   | 2024-01-02 |


## 示例 1：`ROW_NUMBER()` 按部门、销售额排名

```sql
SELECT
  employee,
  department,
  sales,
  ROW_NUMBER() OVER (
    PARTITION BY department
    ORDER BY sales DESC
  ) AS rank_row
FROM sales;
```

### 说明：

* **PARTITION BY department**：每个部门独立排名
* **ORDER BY sales DESC**：销售额高的排前面
* `ROW_NUMBER()`：即使销售额相同，排名也唯一（不跳跃）

查询结果：

| employee | department | sales | rank\_row |
| -------- | ---------- | ----- | --------- |
| Bob      | East       | 200   | 1         |
| David    | East       | 150   | 2         |
| Alice    | East       | 100   | 3         |
| Carol    | West       | 300   | 1         |
| Eva      | West       | 250   | 2         |


## 示例 2：`RANK()` 分部门排名，有跳跃

```sql
SELECT
  employee,
  department,
  sales,
  RANK() OVER (
    PARTITION BY department
    ORDER BY sales DESC
  ) AS rank_rank
FROM sales;
```

🔽 示例结果（假设有并列）：

| employee | department | sales | rank\_rank |
| -------- | ---------- | ----- | ---------- |
| Bob      | East       | 200   | 1          |
| David    | East       | 200   | 1          |
| Alice    | East       | 100   | 3          |
| Carol    | West       | 300   | 1          |
| Eva      | West       | 250   | 2          |

✅ 并列第一（200）后，跳到第 3。


## 示例 3：`DENSE_RANK()` 分部门排名，无跳跃

```sql
SELECT
  employee,
  department,
  sales,
  DENSE_RANK() OVER (
    PARTITION BY department
    ORDER BY sales DESC
  ) AS rank_dense
FROM sales;
```

示例结果：

| employee | department | sales | rank\_dense |
| -------- | ---------- | ----- | ----------- |
| Bob      | East       | 200   | 1           |
| David    | East       | 200   | 1           |
| Alice    | East       | 100   | 2           |
| Carol    | West       | 300   | 1           |
| Eva      | West       | 250   | 2           |

✅ 并列第一后，紧接着排第二，不跳号。


## 示例 4：`SUM()` 的窗口版本（累计销售额）

```sql
SELECT
  employee,
  department,
  sales,
  SUM(sales) OVER (
    PARTITION BY department
    ORDER BY sale_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_sales
FROM sales;
```

### 说明：

* 累计每个部门从最早销售到当前行的**销售总额**
* 按时间累加
* `UNBOUNDED PRECEDING AND CURRENT ROW`：表示从窗口起始到当前行




##  5.3 日期处理函数（以 MySQL 为例）

| 函数               | 作用              | 示例                                      |
| ---------------- | --------------- | --------------------------------------- |
| `NOW()`          | 当前日期时间          | `SELECT NOW();`                         |
| `CURDATE()`      | 当前日期            | `SELECT CURDATE();`                     |
| `YEAR(date)`     | 提取年份            | `YEAR(birthday)`                        |
| `MONTH(date)`    | 提取月份            | `MONTH(join_date)`                      |
| `DATEDIFF(a, b)` | a 与 b 日期差（单位：天） | `DATEDIFF('2025-06-22', '2025-06-01')`  |
| `DATE_ADD()`     | 日期加时间间隔         | `DATE_ADD(NOW(), INTERVAL 7 DAY)`       |
| `STR_TO_DATE()`  | 字符串转日期          | `STR_TO_DATE('2025-06-22', '%Y-%m-%d')` |


## 5.4 字符串处理函数

| 函数                     | 作用                | 示例                                         |
| ---------------------- | ----------------- | ------------------------------------------ |
| `CONCAT(a, b)`         | 字符串拼接             | `CONCAT(first_name, ' ', last_name)`       |
| `SUBSTRING(str, m, n)` | 提取子串              | `SUBSTRING('abcdef', 2, 3)` → `'bcd'`      |
| `LENGTH(str)`          | 字节长度（UTF-8 中文为 3） | `LENGTH('张三')` → `6`                       |
| `CHAR_LENGTH(str)`     | 字符长度              | `CHAR_LENGTH('张三')` → `2`                  |
| `TRIM(str)`            | 去除前后空格            | `TRIM(' abc ')` → `'abc'`                  |
| `UPPER(str)`           | 转大写               | `UPPER('abc')` → `'ABC'`                   |
| `LOWER(str)`           | 转小写               | `LOWER('ABC')` → `'abc'`                   |
| `REPLACE(str, a, b)`   | 替换字符串 a → b       | `REPLACE('abcabc', 'a', 'x')` → `'xbcxbc'` |


